In [ ]:
# Cell 1: Environment Setup and Data Loading
import numpy as np
import pandas as pd

# Load Iranian Industrial Tariff Data directly from repository
data_url = "https://raw.githubusercontent.com/amirkabirian/Hybrid-Quantum-Energy-Optimization/main/data/iran_industrial_energy_data.csv"
tariff_df = pd.read_csv(data_url)

print("Dataset successfully loaded:")
print(tariff_df.head(10))

In [ ]:
# Cell 2: Multi-Industry Presets & QUBO Matrix Construction
INDUSTRY_PROFILES = {
    "Steel_Plant": {
        "description": "Heavy Metallurgy & Electric Arc Furnace",
        "power_specs": [35000, 10000, 5000]
    },
    "Petrochemical_Complex": {
        "description": "Chemical Processing & Heavy Compressor",
        "power_specs": [15000, 8000, 4000]
    },
    "General_Manufacturing": {
        "description": "Flexible Modular Industrial Line",
        "power_specs": [350, 200, 150]
    }
}

# Select Industry Profile
selected_industry = "Steel_Plant"
power_specs = INDUSTRY_PROFILES[selected_industry]["power_specs"]
num_machines = len(power_specs)
num_slots = 3
N = num_machines * num_slots

# Selected hours: Off-Peak (Hr 3), Mid-Peak (Hr 9), On-Peak (Hr 14)
slot_hours = [3, 9, 14]
tariffs = tariff_df.loc[tariff_df['hour'].isin(slot_hours), 'tariff_rate_irr_per_kwh'].values

# Initialize QUBO Matrix Q
Q = np.zeros((N, N))

# 1. Linear terms: Energy Cost Objective
for m in range(num_machines):
    for t in range(num_slots):
        idx = m * num_slots + t
        Q[idx, idx] += power_specs[m] * tariffs[t]

# 2. Hard Constraint: Each machine must run EXACTLY once across slots
lambda_1 = 50000000.0  # Penalty factor for hard constraint
for m in range(num_machines):
    for t1 in range(num_slots):
        idx1 = m * num_slots + t1
        Q[idx1, idx1] -= lambda_1
        for t2 in range(num_slots):
            idx2 = m * num_slots + t2
            Q[idx1, idx2] += lambda_1

print(f"\nQUBO Matrix constructed for [{selected_industry}]. Shape: {Q.shape}")
np.save("qubo_matrix.npy", Q)
print("QUBO Matrix saved to 'qubo_matrix.npy'.")